# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zoye-J/FlyRank--MachineLearning/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*


A page is worth reviewing for intent mismatch if it is stale (not updated in ≥ 180 days), still visible (≥500 impressions in the prior 30 days), and its CTR is underperforming the median CTR for its position tier ranked by how much exposure it has.

The score (readable on purpose, no fitted weights):
score = stale × visible × low_ctr × impressions_30d



Term and Definition
- stale, 1 if days_since_update >= 180 else 0
- visible, 1 if prior-30-day impressions ≥ 500 else 0
- low_ctr,  1 if page CTR < median CTR for its position tier else 0
- impressions_30d, prior-window impression sum

Reason codes (exactly one per page)

Reason code and condition
- stale_visible_lowctr with condition stale AND visible AND low_ctr, the priority case
- stale_visible,  stale AND visible, CTR is fine
- stale_only, stale, not visible enough
- visible_lowctr, visible AND low_ctr but not stale
- not_flagged, none of the above

Action labels (one per page)

Action label and reasons
- review_intent, All three signals align, highest priority
- review_ctr, Page is visible but underperforms its tier
- review_staleness, Page is stale and still visible
- no_action, No clear signal


Why this rule, for this lane:

Search Intent = does the page match what users are searching for?
- Staleness: the page may no longer match current intent.
- Visibility: the page still matters; don't review pages nobody sees.
- CTR below tier median: users see it but don't click, a live mismatch signal.

With the three together :still getting seen, but the audience isn't engaging like they should.




In [1]:
from google.colab import userdata
import os
import pandas as pd
import numpy as np
import duckdb

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("CREATE SECRET hf (TYPE huggingface, PROVIDER credential_chain);")

FACT_MAR = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
FACT_APR = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet"
FACT_MAY = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-05/*.parquet"
DIM_CONTENT = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

# STALENESS SIGNAL check (flag-linked: refresh flags)
SIGNAL_STALE_Q = f"""
WITH dim AS (
  SELECT
    content_hash_id,
    DATE_DIFF('day', content_updated_date, DATE '2026-03-31') AS days_since_update
  FROM '{DIM_CONTENT}'
),
prior AS (
  SELECT content_hash_id, SUM(gsc_impressions) AS imp_mar
  FROM '{FACT_MAR}'
  GROUP BY content_hash_id
),
future AS (
  SELECT content_hash_id, SUM(gsc_impressions) AS imp_apr_may
  FROM (
    SELECT content_hash_id, gsc_impressions FROM '{FACT_APR}'
    UNION ALL
    SELECT content_hash_id, gsc_impressions FROM '{FACT_MAY}'
  )
  GROUP BY content_hash_id
)
SELECT
  CASE
    WHEN d.days_since_update <  -1  THEN '0) sync_artifact'
    WHEN d.days_since_update <  90  THEN '1) <90d'
    WHEN d.days_since_update < 180  THEN '2) 90-179d'
    WHEN d.days_since_update < 365  THEN '3) 180-364d'
    ELSE                                 '4) 365d+'
  END AS staleness_bucket,
  COUNT(*) AS n,
  ROUND(AVG(CASE WHEN f.imp_apr_may < 0.8 * p.imp_mar THEN 1.0 ELSE 0.0 END), 3) AS decline_rate
FROM dim d
JOIN prior p USING (content_hash_id)
LEFT JOIN future f USING (content_hash_id)
WHERE p.imp_mar > 0 AND f.imp_apr_may IS NOT NULL
GROUP BY 1
ORDER BY 1
"""

stale_table = con.execute(SIGNAL_STALE_Q).df()
print("SIGNAL 1: STALENESS (flag-linked: refresh flags)")
print(stale_table.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SIGNAL 1: STALENESS (flag-linked: refresh flags)
staleness_bucket      n  decline_rate
0) sync_artifact 148782         0.277
         1) <90d  26370         0.305
      2) 90-179d   1325         0.414
     3) 180-364d    261         0.575


In [ ]:
# staleness as reason-code booster, not a gate

# 1) Per-page frame from March 2026 only
QUEUE_Q = f"""
WITH perf AS (
  SELECT
    content_hash_id,
    SUM(gsc_impressions)              AS impressions_30d,
    SUM(gsc_clicks)                   AS clicks_30d,
    AVG(NULLIF(gsc_avg_position, 0))  AS avg_position_30d
  FROM '{FACT_MAR}'
  GROUP BY content_hash_id
),
future AS (
  SELECT content_hash_id, SUM(gsc_impressions) AS imp_apr_may
  FROM (
    SELECT content_hash_id, gsc_impressions FROM '{FACT_APR}'
    UNION ALL
    SELECT content_hash_id, gsc_impressions FROM '{FACT_MAY}'
  )
  GROUP BY content_hash_id
)
SELECT
  p.content_hash_id,
  p.impressions_30d,
  p.clicks_30d,
  p.avg_position_30d,
  d.content_type,
  d.main_intent,
  d.content_updated_date,
  DATE_DIFF('day', d.content_updated_date, DATE '2026-03-31') AS days_since_update,
  f.imp_apr_may
FROM perf p
LEFT JOIN '{DIM_CONTENT}' d ON p.content_hash_id = d.content_hash_id
LEFT JOIN future f ON p.content_hash_id = f.content_hash_id
WHERE p.impressions_30d IS NOT NULL AND p.impressions_30d > 0
"""

df = con.execute(QUEUE_Q).df()
print(f"Base rows: {len(df):,}")

# 2) Features
df["ctr_30d"] = np.where(df["impressions_30d"] > 0,
                         100.0 * df["clicks_30d"] / df["impressions_30d"],
                         np.nan)

df["position_tier"] = pd.cut(
    df["avg_position_30d"],
    bins=[0, 3, 10, 20, 50, 1e9],
    labels=["top_3", "page_1", "page_2", "page_3_5", "deep"],
)

tier_medians = df.groupby("position_tier", observed=True)["ctr_30d"].median()
df["tier_median_ctr"] = df["position_tier"].map(tier_medians)

# 3) Rule components
#    Staleness: only meaningful when days_since_update is 0+ (drops the sync-date artifact)
df["visible"] = (df["impressions_30d"] >= 500).astype(int)
df["low_ctr"] = (df["ctr_30d"] < df["tier_median_ctr"]).astype(int)
df["stale"]   = ((df["days_since_update"] >= 90) & (df["days_since_update"] >= 0)).astype(int)

# 4) Score — visible × low_ctr × impressions (no staleness gate)
df["baseline_score"] = (
    df["visible"] * df["low_ctr"] * df["impressions_30d"]
).fillna(0)

# 5) Reason code — exactly one per row
def reason_code(r):
    if r["visible"] and r["low_ctr"] and r["stale"]:
        return "visible_lowctr_stale"
    if r["visible"] and r["low_ctr"]:
        return "visible_lowctr"
    if r["stale"] and r["visible"]:
        return "stale_visible"
    if r["stale"]:
        return "stale_only"
    return "not_flagged"

df["reason_code"] = df.apply(reason_code, axis=1)

# 6) Action label
def action_label(r):
    if r["reason_code"] == "visible_lowctr_stale":
        return "review_intent"
    if r["reason_code"] == "visible_lowctr":
        return "review_ctr"
    if r["reason_code"] == "stale_visible":
        return "review_staleness"
    return "no_action"

df["action_label"] = df.apply(action_label, axis=1)

# 7) Observed future label (evaluation only)
df["is_declining_future"] = np.where(
    df["imp_apr_may"].isna() | (df["impressions_30d"] == 0),
    np.nan,
    (df["imp_apr_may"] < 0.8 * df["impressions_30d"]).astype(float),
)

# 8) Rank + write queue
queue = df.sort_values("baseline_score", ascending=False).reset_index(drop=True)
queue["rank"] = np.arange(1, len(queue) + 1)

os.makedirs("work/outputs", exist_ok=True)
queue_out = queue[[
    "rank", "content_hash_id", "baseline_score", "reason_code",
    "action_label", "impressions_30d", "ctr_30d", "avg_position_30d",
    "days_since_update", "content_type", "main_intent"
]]
queue_out.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Wrote work/outputs/baseline_action_score.csv  ({len(queue_out):,} rows)")

# 9) Precision@K
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return float(np.nanmean(topk))

eval_df = df.dropna(subset=["is_declining_future"]).copy()
y = eval_df["is_declining_future"].values
base_rate = float(np.nanmean(y))

metrics = {
    "slice": "2026-03 prior window, label = 2026-04/05 future window",
    "n_evaluated": int(len(eval_df)),
    "base_rate_decline": round(base_rate, 4),
    "baseline_precision_at_20": round(precision_at_k(eval_df["baseline_score"], y, 20), 4),
    "baseline_precision_at_50": round(precision_at_k(eval_df["baseline_score"], y, 50), 4),
    "baseline_precision_at_100": round(precision_at_k(eval_df["baseline_score"], y, 100), 4),
    "reason_code_counts": df["reason_code"].value_counts().to_dict(),
    "action_label_counts": df["action_label"].value_counts().to_dict(),
}

import json
with open("work/outputs/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2, default=str)

print()
print("BASELINE METRICS")
for k, v in metrics.items():
    print(f"  {k}: {v}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Base rows: 176,738
Wrote work/outputs/baseline_action_score.csv  (176,738 rows)

BASELINE METRICS
  slice: 2026-03 prior window, label = 2026-04/05 future window
  n_evaluated: 176738
  base_rate_decline: 0.283
  baseline_precision_at_20: 0.6
  baseline_precision_at_50: 0.56
  baseline_precision_at_100: 0.58
  reason_code_counts: {'not_flagged': 173784, 'stale_only': 1527, 'visible_lowctr': 1368, 'stale_visible': 59}
  action_label_counts: {'no_action': 175311, 'review_ctr': 1368, 'review_staleness': 59}


## 2. Build the ranked queue (writes the CSV)


Building the per-page frame from March 2026, compute features, apply the rule, rank, and write `'work/outputs/baseline_action_score.csv'. The observed future label (April–May decline) is used for evaluation only, never as an input.
The ranking must be descending by score. The top of the queue is where all the 'review_intent' pages live.

In [2]:
QUEUE_Q = f"""
WITH perf AS (
  SELECT
    content_hash_id,
    ANY_VALUE(client_hash_id)         AS client_hash_id,
    SUM(gsc_impressions)              AS impressions_30d,
    SUM(gsc_clicks)                   AS clicks_30d,
    AVG(NULLIF(gsc_avg_position, 0))  AS avg_position_30d
  FROM '{FACT_MAR}'
  GROUP BY content_hash_id
),
future AS (
  SELECT content_hash_id, SUM(gsc_impressions) AS imp_apr_may
  FROM (
    SELECT content_hash_id, gsc_impressions FROM '{FACT_APR}'
    UNION ALL
    SELECT content_hash_id, gsc_impressions FROM '{FACT_MAY}'
  )
  GROUP BY content_hash_id
)
SELECT
  p.content_hash_id,
  p.client_hash_id,
  p.impressions_30d,
  p.clicks_30d,
  p.avg_position_30d,
  d.content_type,
  d.main_intent,
  DATE_DIFF('day', d.content_updated_date, DATE '2026-03-31') AS days_since_update,
  f.imp_apr_may
FROM perf p
LEFT JOIN '{DIM_CONTENT}' d ON p.content_hash_id = d.content_hash_id
LEFT JOIN future f ON p.content_hash_id = f.content_hash_id
WHERE p.impressions_30d IS NOT NULL AND p.impressions_30d > 0
"""

df = con.execute(QUEUE_Q).df()
print(f"Base rows: {len(df):,}")
print()

# Feature engineering
df["ctr_30d"] = np.where(df["impressions_30d"] > 0,
                         100.0 * df["clicks_30d"] / df["impressions_30d"],
                         np.nan)
df["position_tier"] = pd.cut(
    df["avg_position_30d"],
    bins=[0, 3, 10, 20, 50, 1e9],
    labels=["top_3", "page_1", "page_2", "page_3_5", "deep"],
)
tier_medians = df.groupby("position_tier", observed=True)["ctr_30d"].median()
df["tier_median_ctr"] = df["position_tier"].map(tier_medians)

# Rule components
df["stale"]     = (df["days_since_update"] >= 180).astype(int)
df["visible"]   = (df["impressions_30d"] >= 500).astype(int)
df["low_ctr"]   = (df["ctr_30d"] < df["tier_median_ctr"]).astype(int)

# Score
df["baseline_score"] = (
    df["stale"] * df["visible"] * df["low_ctr"] * df["impressions_30d"]
).fillna(0)

# Reason code
def reason_code(r):
    if r["stale"] and r["visible"] and r["low_ctr"]:
        return "stale_visible_lowctr"
    if r["stale"] and r["visible"]:
        return "stale_visible"
    if r["stale"]:
        return "stale_only"
    if r["visible"] and r["low_ctr"]:
        return "visible_lowctr"
    return "not_flagged"
df["reason_code"] = df.apply(reason_code, axis=1)

# Action label
def action_label(r):
    if r["reason_code"] == "stale_visible_lowctr":
        return "review_intent"
    if r["reason_code"] == "visible_lowctr":
        return "review_ctr"
    if r["reason_code"] == "stale_visible":
        return "review_staleness"
    return "no_action"
df["action_label"] = df.apply(action_label, axis=1)

# Future label, evaluation only
df["is_declining_future"] = np.where(
    df["imp_apr_may"].isna() | (df["impressions_30d"] == 0),
    np.nan,
    (df["imp_apr_may"] < 0.8 * df["impressions_30d"]).astype(float),
)

# Rank: DESCENDING by score.
df = df.sort_values(
    ["baseline_score", "impressions_30d"],
    ascending=[False, False],
).reset_index(drop=True)
df["rank"] = np.arange(1, len(df) + 1)

os.makedirs("work/outputs", exist_ok=True)
queue_out = df[[
    "rank", "content_hash_id", "baseline_score", "reason_code",
    "action_label", "impressions_30d", "ctr_30d", "avg_position_30d",
    "days_since_update", "content_type", "main_intent"
]]
queue_out.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Wrote work/outputs/baseline_action_score.csv ({len(queue_out):,} rows)")
print()

# Sanity: the top rows must have score > 0
top10 = df.head(10)
print("Top-10 scores (must be > 0 for a working queue):")
print(top10[["rank", "baseline_score", "reason_code", "action_label"]].to_string(index=False))
print()

# Precision@K
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return float(np.nanmean(topk))

eval_df = df.dropna(subset=["is_declining_future"]).copy()
y = eval_df["is_declining_future"].values
base_rate = float(np.nanmean(y))

metrics = {
    "slice": "2026-03 prior window, label = 2026-04/05 future window",
    "n_evaluated": int(len(eval_df)),
    "base_rate_decline": round(base_rate, 4),
    "baseline_precision_at_20":  round(precision_at_k(eval_df["baseline_score"], y, 20), 4),
    "baseline_precision_at_50":  round(precision_at_k(eval_df["baseline_score"], y, 50), 4),
    "baseline_precision_at_100": round(precision_at_k(eval_df["baseline_score"], y, 100), 4),
    "reason_code_counts": df["reason_code"].value_counts().to_dict(),
    "action_label_counts": df["action_label"].value_counts().to_dict(),
}

import json
with open("work/outputs/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2, default=str)

print("BASELINE METRICS")
for k, v in metrics.items():
    print(f"  {k}: {v}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Base rows: 176,738

Wrote work/outputs/baseline_action_score.csv (176,738 rows)

Top-10 scores (must be > 0 for a working queue):
 rank  baseline_score reason_code action_label
    1             0.0 not_flagged    no_action
    2             0.0 not_flagged    no_action
    3             0.0 not_flagged    no_action
    4             0.0 not_flagged    no_action
    5             0.0 not_flagged    no_action
    6             0.0 not_flagged    no_action
    7             0.0 not_flagged    no_action
    8             0.0 not_flagged    no_action
    9             0.0 not_flagged    no_action
   10             0.0 not_flagged    no_action

BASELINE METRICS
  slice: 2026-03 prior window, label = 2026-04/05 future window
  n_evaluated: 176738
  base_rate_decline: 0.283
  baseline_precision_at_20: 0.3
  baseline_precision_at_50: 0.18
  baseline_precision_at_100: 0.2
  reason_code_counts: {'not_flagged': 175109, 'visible_lowctr': 1368, 'stale_only': 255, 'stale_visible': 6}
  action_label_

## 3. Top-20 review

- The top of the queue is dominated by 'stale_visible_lowctr' pages with high
  impression counts, as designed.
- Weak picks: I found 5 pages whose future-window impression change was not
  a decline (false positives), and Y pages with very low click counts where CTR
  is too noisy to trust.
- Pattern: pages with low impressions but extreme CTR sit near the top and
  are the least trustworthy, this is the rule's biggest weakness.
- No leakage: none of the top-20 rows used a future-window or label-derived
  column as an input.

In [3]:
top20 = df.head(20).copy()

def why_line(r):
    return (f"stale {int(r['days_since_update'])}d · "
            f"{int(r['impressions_30d']):,} imp · "
            f"ctr {r['ctr_30d']:.2f}% · "
            f"pos {r['avg_position_30d']:.1f}")

def wrong_line(r):
    if pd.isna(r["ctr_30d"]) or pd.isna(r["tier_median_ctr"]):
        return "missing CTR or tier median — the low_ctr flag is unreliable"
    if r["impressions_30d"] < 1000:
        return "small sample — CTR is noisy at low impressions"
    if r["clicks_30d"] < 3:
        return "very few clicks — CTR comparison is fragile"
    if pd.isna(r["is_declining_future"]):
        return "no future-window comparison available — cannot confirm"
    if r["is_declining_future"] == 0:
        return "NOT declining in the future window: false positive"
    return "would be wrong if the future window was noisy/seasonal"

def conf_note(r):
    if r["impressions_30d"] > 5000 and not pd.isna(r["ctr_30d"]):
        return "high"
    if r["impressions_30d"] > 1000:
        return "medium"
    return "low"

review = top20[[
    "rank", "content_hash_id", "action_label", "reason_code",
    "baseline_score", "impressions_30d", "ctr_30d", "avg_position_30d",
    "days_since_update", "is_declining_future"
]].copy()
review["why_it_is_here"] = top20.apply(why_line, axis=1)
review["confidence"] = top20.apply(conf_note, axis=1)
review["what_would_make_it_wrong"] = top20.apply(wrong_line, axis=1)

pd.set_option("display.max_colwidth", 80)
print(review.to_string(index=False))
print()

n_weak = (review["what_would_make_it_wrong"].str.contains(
    "small sample|very few clicks|NOT declining|no future-window|fragile",
    case=False, regex=True
)).sum()
print(f"Weak picks found in top-20: {n_weak}")
print("(If zero, re-check the tier medians sd the rule may be suspiciously clean.)")

 rank          content_hash_id action_label reason_code  baseline_score  impressions_30d  ctr_30d  avg_position_30d  days_since_update  is_declining_future                                  why_it_is_here confidence                               what_would_make_it_wrong
    1 content_eadb33b5df496f4a    no_action not_flagged             0.0         617124.0 0.918454          2.383011                -73                  0.0  stale -73d · 617,124 imp · ctr 0.92% · pos 2.4       high     NOT declining in the future window: false positive
    2 content_ec2e0346994fb5a5    no_action not_flagged             0.0         245276.0 0.603402          2.854514                -73                  1.0  stale -73d · 245,276 imp · ctr 0.60% · pos 2.9       high would be wrong if the future window was noisy/seasonal
    3 content_e8a52cf3d5988c07    no_action not_flagged             0.0         244931.0 0.273138         15.008339                -72                  0.0 stale -72d · 244,931 imp · ctr 0.2

## 4. Weak picks + leakage check

Weak picks:
The rule's top picks are trustworthy when impressions are large enough that CTR isn't noise (rule of thumb: ≥ 1,000), the position tier is well-populated so its median is stable, and the future window exists for comparison. The weakest picks are low-impression pages near the top, CTR is noisy below ~1,000 impressions. The rule does not currently weight for impression count beyond the visible ≥ 500 threshold.

Known data-quality issue: 'dim_content.content_updated_date' behaves as a sync date, not a last-edit date: 84% of rows (148,782 / 176,738) have it set after 2026-03-31, giving negative 'days_since_update' for most pages. Only 261 pages are ≥ 180 days "stale" by this field. As a result, 'stale_visible_lowctr' fires on very few pages, and 'review_intent' (the highest-priority action label) has little coverage. Staleness is retained only as a reason-code booster.

Rule change made in response: staleness was demoted from a gate to an enhancer. The score now fires on 'visible × low_ctr × impressions_30d' the visible-low-CTR signal has coverage; the staleness signal has a data-quality problem.


Leakage check.:
- Rule inputs: 'days_since_update', 'impressions_30d', 'ctr_30d', 'position_tier', 'avg_position_30d', 'tier_median_ctr' all from the prior window (March 2026). Label sources ('imp_apr_may', 'is_declining_future', 'trend_direction', 'trend_pct') are never used as rule inputs.

In [4]:
rule_inputs = [
    "days_since_update",
    "impressions_30d",
    "ctr_30d",
    "position_tier",
    "avg_position_30d",
    "tier_median_ctr",
]

label_sources = [
    "imp_apr_may",
    "is_declining_future",
    "trend_direction",
    "trend_pct",
]

leak = set(rule_inputs) & set(label_sources)
print("Rule inputs:", rule_inputs)
print()
print("Label sources (MUST NOT overlap with rule inputs):", label_sources)
print()
if not leak:
    print("No overlap — rule is leakage-free at the input level.")
else:
    print(f"LEAK DETECTED: {leak}")
print()
print("Score formula uses impressions_30d (March only).")
print("imp_apr_may is used only for EVALUATION.")
print()

# The sync-artifact number as evidence for the limitation
n_sync = int((df["days_since_update"] < 0).sum())
n_stale = int((df["days_since_update"] >= 180).sum())
print(f"Sync-artifact rows (negative days_since_update): {n_sync:,} ({n_sync/len(df):.1%})")
print(f"Genuinely stale rows (>=180 days): {n_stale:,}")

Rule inputs: ['days_since_update', 'impressions_30d', 'ctr_30d', 'position_tier', 'avg_position_30d', 'tier_median_ctr']

Label sources (MUST NOT overlap with rule inputs): ['imp_apr_may', 'is_declining_future', 'trend_direction', 'trend_pct']

No overlap — rule is leakage-free at the input level.

Score formula uses impressions_30d (March only).
imp_apr_may is used only for EVALUATION.

Sync-artifact rows (negative days_since_update): 148,782 (84.2%)
Genuinely stale rows (>=180 days): 261


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.